# Multimodal PDF RAG with Weaviate & Google Gemini

This notebook demonstrates how to process and query PDF documents using Weaviate & Google Gemini. We will:
1. Download a complex 50-page PDF document from the [Document Haystack dataset](https://huggingface.co/datasets/AmazonScience/document-haystack).
2. Convert the PDF pages into images.
3. Ingest the images into Weaviate using the `multi2vec_google_gemini` vectorizer powered by Gemini Embedding 2, Google’s first fully multimodal mbedding model.
4. Perform multimodal RAG with Weaviate’s `generative-google` module using Gemini 3 Flash model.

In [ ]:
# Setup and install necessary dependencies
!pip install -q weaviate-client pdf2image pillow requests huggingface_hub

### Step 1: Connect to Weaviate

First, set up a Weaviate instance and connect to it. If you don’t already have one, sign up for a free Weaviate Cloud sandbox cluster [here](https://console.weaviate.cloud/signin?utm_source=github&utm_campaign=gemini_recipe).

Once your cluster is ready:

1. Copy your cluster URL and generate an API key.
2. Add them as environment variables named `WEAVIATE_URL` and `WEAVIATE_API_KEY`.
3. Go to [Google AI Studio](https://aistudio.google.com/api-keys), generate a Google API key, and save it as `GOOGLE_API_KEY` environment variable.

In [ ]:
import os
import base64
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.config import Configure, Property, DataType
from PIL import Image
from io import BytesIO

WEAVIATE_URL = os.getenv("WEAVIATE_URL")
WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
    headers={
        "X-Goog-Api-Key": GOOGLE_API_KEY,
    }
)

print(f"Connected to Weaviate: {client.is_ready()}")

### Step 2: Create Weaviate Collection

In [ ]:
COLLECTION_NAME = "DocumentHaystack"

if client.collections.exists(COLLECTION_NAME):
    print(f"Collection {COLLECTION_NAME} already exists.")
    # Un-comment the following lines to delete the existing collection
    # client.collections.delete(COLLECTION_NAME)
    # print(f"Deleted collection: {COLLECTION_NAME}")
else:
    collection = client.collections.create(
        name=COLLECTION_NAME,
        properties=[
            Property(name="doc_page", data_type=DataType.BLOB),
            Property(name="document_id", data_type=DataType.TEXT),
            Property(name="page_number", data_type=DataType.INT),
        ],
        vector_config=[
            Configure.Vectors.multi2vec_google_gemini(
                name="doc_vector",
                image_fields=["doc_page"],
                model="gemini-embedding-2-preview",
                vector_index_config=Configure.VectorIndex.flat(),
            )
        ],
        generative_config=Configure.Generative.google_gemini(),
    )
    print(f"Created collection: {COLLECTION_NAME}")

### Step 3: Fetch and Process from Document Haystack
We'll download a 50-page document from the Document Haystack dataset on Hugging Face. Then, convert the PDF directly into a series of PIL images.

*Note: `pdf2image` requires poppler to be installed on your system (Mac users can simply run `brew install poppler`).*


In [ ]:
from huggingface_hub import hf_hub_download
from pdf2image import convert_from_path

pdf_path = hf_hub_download(
    repo_id="AmazonScience/Document-Haystack",
    filename="AIG/AIG_50Pages/AIG_50Pages_ImageNeedles.pdf",
    repo_type="dataset",
)

print(f"Downloaded PDF to: {pdf_path}")

# Convert PDF to a list of images (one per page)
pages = convert_from_path(pdf_path, dpi=150)

print(f"Converted PDF into {len(pages)} images.")


def image_to_base64(image):
    buffer = BytesIO()
    image.save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

document_dataset = []
for idx, page_img in enumerate(pages):
    document_dataset.append(
        {
            "document_id": "AIG_50Pages_ImageNeedles",
            "page_number": idx + 1,
            "image": page_img,
        }
    )

### Step 4: Index Documents
We iterate through our processed PDF pages and upload the base64-encoded BLOBs to Weaviate. 
Weaviate embeds the pages using Gemini's embedding API and stores the resulting vectors.

In [ ]:
collection = client.collections.get(COLLECTION_NAME)

In [ ]:
print("Starting ingestion...")
with collection.batch.dynamic() as batch:
    for document in document_dataset:
        img_base64 = image_to_base64(document["image"])
        batch.add_object(
            properties={
                "doc_page": img_base64,
                "document_id": document["document_id"],
                "page_number": document["page_number"],
            },
        )
        print(f"Indexed page {document['page_number']}")

# Check for ingestion errors
if len(collection.batch.failed_objects) > 0:
    print("Some/All objects failed to import.")
else:
    print(f"Successfully indexed all {len(document_dataset)} pages!")

### Step 5: Multimodal RAG with Google Gemini
We can now query Weaviate for specific "needles" hidden in the document. Weaviate will:
1. Embed our text query using Gemini.
2. Search and retrieve the most similar pages using `near_text`.
3. Provide the query and the visual context to Gemini 3 Flash to generate a natural language response.


In [ ]:
import base64
from IPython.display import Image, display
from weaviate.classes.generate import GenerativeConfig, GenerativeParameters

query_text = "What is the secret flower?"

prompt = GenerativeParameters.grouped_task(
    prompt=f"Look at the provided document pages and answer the following question. Remember, the answer might be hidden as a short visual or textual statement: {query_text}",
    image_properties=["doc_page"],
)

response = collection.generate.near_text(
    query=query_text,
    limit=3,
    grouped_task=prompt,
    generative_provider=GenerativeConfig.google_gemini(model="gemini-3-flash-preview"),
    return_properties=["doc_page", "page_number"],
)

print(f"Query Sent: \n{query_text}\n")
print(f"Generated Answer: \n{response.generated}\n")

print("\n--- Retrieved Documents ---")
for i, obj in enumerate(response.objects):
    page_num = obj.properties.get("page_number", "Unknown")
    print(f"\nDocument {i + 1} (Page {page_num}):")
    if "doc_page" in obj.properties:
        img_data = base64.b64decode(obj.properties["doc_page"])
        display(Image(data=img_data, width=400))
    else:
        print("No image returned for this document.")

In [ ]:
client.close()